# TP3 Chatbots basados en recuperación de la información

En inglés information retrieval chatbots

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [14]:
!pip install spacy --quiet
!python -m spacy download es_core_news_sm --quiet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 93.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Actividades



### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

###  2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)

### 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado de las librerías Spacy o nltk o gensim eligió. Justifique.

### 4) Muestra ambos chatbots funcionando (1 punto)

Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

### 5) Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.



### No olvides:

* Explicar tus decisiones y configuraciones. Añadir tus conclusiones.
* Anunciar en el foro cuál será tu aplicación y postear tu entrega y tus avances.
* Debes subir tu notebook a un repo GitHub público de tu propiedad compartido + enlace colab.
* Documentar todo el proceso.
* Citar tus fuentes





# Apartado 1:

## 1.1)

La aplicacion para el chatbot que voy a desarrollar trata sobre atencion al cliente, especificamente en una distribuidora mayorista de articulos de ferreteria.
Elegi especificamente esta aplicacion ya que es un tema sobre el que tengo conocimiento ya que trabajo en FGP, una distribuidora nacional.

El enfoque que le voy a dar es el siguiente:

Todos ellos relacionados con dudas que puedan tener los clientes de la ferreteria.

*   Dudas sobre tiempos de envio
*   Dudas sobre herramientas especificas
*   Dudas sobre stock en una clase de herramientas
*   Dudas sobre como contactar a otras areas
*   Dudas sobre a quien debe acudir para consultar deudas
*   Informacion general de la empresa










## 1.2)

El diccionario pregunta-respuesta consta de 24 preguntas con sus respectivas respuestas que abordan los temas definidos en el alcance del punto 1.1

In [15]:
import pandas as pd

#Prompt: teniendo en cuenta el alcance que le quiero dar al chatbot, definido en el apartado 1.1, podrias generar un diccionario de minimo 20 preguntas?
dataset = [
    ("¿Cuánto tarda en llegar un pedido?",
     "Los pedidos confirmados antes de las 12 h se preparan en el día. Dentro de la zona de reparto propio llegan en 24 a 72 h hábiles; al resto del país se despachan por expreso y demoran entre 3 y 7 días hábiles según la localidad."),
    ("¿Hacen envíos al interior del país?",
     "Sí, enviamos a todo el país. Fuera de la zona de reparto propio la mercadería viaja por el expreso que elija el cliente, y el flete corre por su cuenta."),
    ("¿Cómo puedo saber dónde está mi pedido?",
     "Consultá con tu vendedor o con el área de Logística indicando el número de pedido. Si ya fue despachado, te pasamos el número de guía del transporte para que lo sigas."),
    ("¿Qué días pasa el reparto por mi zona?",
     "El reparto propio tiene días fijos para cada zona. Tu vendedor te pasa el cronograma; si el pedido entra después del cierre, sale en el reparto siguiente."),
    ("¿Qué diferencia hay entre un taladro y un rotomartillo?",
     "El taladro sirve para perforar madera, metal y mampostería liviana. El rotomartillo tiene un mecanismo de golpe más potente y usa mechas SDS, por eso es el indicado para hormigón y piedra."),
    ("¿Qué disco uso para cortar metal con la amoladora?",
     "Para metal se usa un disco abrasivo de corte fino (por ejemplo 115 x 1 mm). Para acero inoxidable hay discos específicos, y los diamantados se usan para cerámica, ladrillo y hormigón."),
    ("¿Qué mecha necesito para perforar hormigón?",
     "Una mecha de widia (punta de metal duro) con taladro percutor, o una mecha SDS Plus si trabajás con rotomartillo."),
    ("¿Las herramientas eléctricas tienen garantía?",
     "Sí, tienen garantía oficial del fabricante, generalmente de 6 a 12 meses según la marca. Para usarla hace falta la factura de compra."),
    ("¿Tienen stock de herramientas eléctricas?",
     "Sí, trabajamos con stock permanente de taladros, amoladoras, sierras y atornilladores. La disponibilidad exacta de cada artículo te la confirma tu vendedor."),
    ("¿Hay stock de tornillos y bulones?",
     "Sí, la tornillería y bulonería en medidas estándar tiene stock permanente. Las medidas especiales se piden a fábrica."),
    ("¿Qué pasa si un producto que pedí está sin stock?",
     "Queda como pendiente y se envía con tu próxima compra cuando ingrese. Si lo necesitás ya, tu vendedor te puede ofrecer un reemplazo equivalente de otra marca."),
    ("¿Cuándo vuelve a ingresar mercadería?",
     "Depende de cada proveedor, pero en general la reposición es semanal. Tu vendedor te puede dar la fecha estimada de un artículo puntual."),
    ("¿Cómo me comunico con mi vendedor?",
     "Por WhatsApp o teléfono al número de tu vendedor asignado. Si no lo tenés, llamá a la central y te derivan."),
    ("¿Cómo hago un reclamo por mercadería faltante o dañada?",
     "Comunicate con el área de Logística dentro de las 48 h de recibido el pedido, con el número de remito y fotos de la mercadería."),
    ("¿Con quién hablo para cambiar la dirección de entrega?",
     "Con el área de Logística o con tu vendedor, siempre antes de que el pedido sea despachado."),
    ("¿Cómo hago para devolver un producto?",
     "Las devoluciones se coordinan con tu vendedor. Una vez recibida y revisada la mercadería, el área administrativa emite la nota de crédito."),
    ("¿Dónde consulto el saldo de mi cuenta corriente?",
     "En el área de Cobranzas. Pediles el resumen de cuenta y te lo envían por mail con las facturas pendientes y sus vencimientos."),
    ("¿A quién le pago una factura vencida?",
     "Al área de Cobranzas, que te indica los medios de pago disponibles: transferencia, cheque o cobrador."),
    ("¿Por qué tengo la cuenta bloqueada?",
     "Las cuentas se bloquean cuando hay facturas vencidas o se supera el límite de crédito. Contactá a Cobranzas para regularizar y que se liberen tus pedidos."),
    ("¿Cómo informo un pago que ya hice?",
     "Enviá el comprobante a Cobranzas indicando tu razón social y qué facturas cancela ese pago."),
    ("¿Cuál es el horario de atención?",
     "Atendemos de lunes a viernes de 8 a 17 h y los sábados de 8 a 12 h."),
    ("¿Venden a consumidor final?",
     "No, somos una distribuidora mayorista: vendemos solo a comercios como ferreterías, corralones y pinturerías."),
    ("¿Cómo me registro como cliente?",
     "Contactá a un vendedor con tu CUIT y los datos del comercio. Se da de alta la cuenta y se te asigna un vendedor para tu zona."),
    ("¿Qué productos venden?",
     "Herramientas manuales y eléctricas, tornillería y bulonería, electricidad, plomería, jardinería y artículos de ferretería en general."),
]

#creacion del dataframe
df = pd.DataFrame(dataset, columns=["pregunta", "respuesta"])

#le asigno una variable a cada columna
preguntas = df["pregunta"].tolist()
respuestas = df["respuesta"].tolist()


df

,pregunta,respuesta
0,¿Cuánto tarda en llegar un pedido?,Los pedidos confirmados antes de las 12 h se p...
1,¿Hacen envíos al interior del país?,"Sí, enviamos a todo el país. Fuera de la zona ..."
2,¿Cómo puedo saber dónde está mi pedido?,Consultá con tu vendedor o con el área de Logí...
3,¿Qué días pasa el reparto por mi zona?,El reparto propio tiene días fijos para cada z...
4,¿Qué diferencia hay entre un taladro y un roto...,"El taladro sirve para perforar madera, metal y..."
5,¿Qué disco uso para cortar metal con la amolad...,Para metal se usa un disco abrasivo de corte f...
6,¿Qué mecha necesito para perforar hormigón?,Una mecha de widia (punta de metal duro) con t...
7,¿Las herramientas eléctricas tienen garantía?,"Sí, tienen garantía oficial del fabricante, ge..."
8,¿Tienen stock de herramientas eléctricas?,"Sí, trabajamos con stock permanente de taladro..."
9,¿Hay stock de tornillos y bulones?,"Sí, la tornillería y bulonería en medidas está..."


# Apartado 2:

## Chatbot creado usando TF-IDF

Como las librerias necesarias ya estan importadas, voy a comenzar vectorizando cada pregunta del dataset.
Luego, se vectoriza cada pregunta que realice un usuario y usando la similitud del coseno se la relaciona con la de mayor similitud dentro del dataset.
Una vez finalizada esa relacion, se le brinda la respuesta de la pregunta con mayor coincidencia al usuario.

In [16]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(preguntas)
print(tfidf_matrix.shape)

(24, 98)


La funcion .shape me indica que mi matriz tfidf, consta de 24 filas (preguntas) y 98 columnas, osea 98 palabras unicas que encontro en todas preguntas.

In [17]:
def chatbot_tfidf(consulta_usuario):
    consulta_vec = vectorizer.transform([consulta_usuario]) #se vectoriza la consulta realizada por el usuario
    similitudes = cosine_similarity(consulta_vec, tfidf_matrix).flatten() #se calcula la similitud coseno de la consulta con las preguntas predefinidas
    indice = similitudes.argmax() #busca en donde esta la pregunta predefinida con mayor similitud
    print(f"Pregunta más parecida: {preguntas[indice]} (similitud: {similitudes[indice]:.2f})")
    return respuestas[indice]


#prueba
consulta = "cuando demora en llegar mi pedido?"
print(chatbot_tfidf(consulta))

Pregunta más parecida: ¿Cuánto tarda en llegar un pedido? (similitud: 0.67)
Los pedidos confirmados antes de las 12 h se preparan en el día. Dentro de la zona de reparto propio llegan en 24 a 72 h hábiles; al resto del país se despachan por expreso y demoran entre 3 y 7 días hábiles según la localidad.


In [24]:
#Prompt: Podrias crear un ciclo para probar el funcionamiento del chatbot
while True:
    entrada_usuario = input("Tú: ")
    if entrada_usuario.lower() == "salir":
        print("Chatbot: Hasta luego.")
        break
    print("Chatbot:", chatbot_tfidf(entrada_usuario))

Tú: que herramienta puedo utilizar para cortar hierro
Pregunta más parecida: ¿Qué disco uso para cortar metal con la amoladora? (similitud: 0.32)
Chatbot: Para metal se usa un disco abrasivo de corte fino (por ejemplo 115 x 1 mm). Para acero inoxidable hay discos específicos, y los diamantados se usan para cerámica, ladrillo y hormigón.
Tú: salir
Chatbot: Hasta luego.


# Apartado 3

El modelo que seleccione para el chatbot que utiliza embeddings es `es_core_news_md` de la libreria spaCy ya que trae vectores en español.

In [19]:
!python -m spacy download es_core_news_md --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 13.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [20]:
import spacy
import numpy as np

nlp = spacy.load("es_core_news_md")

# Cada pregunta se convierte en un vector de 300 dimensiones
# (spaCy calcula el promedio de los vectores de sus palabras)
embeddings_preguntas = np.array([nlp(p).vector for p in preguntas])

print(embeddings_preguntas.shape)

(24, 300)


El resultado (24, 300) indica que cada una de las 24 preguntas del dataset quedó representada por un vector de 300 dimensiones. Cada palabra ya tiene su propio vector de 300 números en el modelo pre-entrenado, y spaCy los promedia para obtener un único vector por pregunta.


In [21]:
def chatbot_embeddings(consulta_usuario):
    consulta_vec = nlp(consulta_usuario).vector.reshape(1, -1) #se vectoriza la consulta con el embedding y se pasa a formato matriz (1 fila)
    similitudes = cosine_similarity(consulta_vec, embeddings_preguntas).flatten() #se calcula la similitud coseno de la consulta con las preguntas predefinidas
    indice = similitudes.argmax()   #busca dónde está la pregunta predefinida con mayor similitud
    print(f"Pregunta más parecida: {preguntas[indice]} (similitud: {similitudes[indice]:.2f})")
    return respuestas[indice]


#misma consulta que en el chat tfidf
consulta = "cuando demora en llegar mi pedido?"
print(chatbot_embeddings(consulta))

Pregunta más parecida: ¿Dónde consulto el saldo de mi cuenta corriente? (similitud: 0.78)
En el área de Cobranzas. Pediles el resumen de cuenta y te lo envían por mail con las facturas pendientes y sus vencimientos.


In [25]:
#mismo ciclo while utilizado en el chatbot tfidf
while True:
    entrada_usuario = input("Tú: ")
    if entrada_usuario.lower() == "salir":
        print("Chatbot: Hasta luego.")
        break
    print("Chatbot:", chatbot_embeddings(entrada_usuario))

Tú: que herramienta puedo utilizar para cortar hierro
Pregunta más parecida: ¿Qué disco uso para cortar metal con la amoladora? (similitud: 0.58)
Chatbot: Para metal se usa un disco abrasivo de corte fino (por ejemplo 115 x 1 mm). Para acero inoxidable hay discos específicos, y los diamantados se usan para cerámica, ladrillo y hormigón.
Tú: salir
Chatbot: Hasta luego.


# Apartado 4:

La lista de preguntas para probar el funcionamiento de los chatbots:

* Tienen tornillos en stock?

* A que hora abren?

* Como hago para pagar una deuda?

* Me pueden mandar el pedido a otra dirección?

* Quien gano el partido de ayer?

* Que mecha uso para agujerear hormigon?

* Cuanto tarda el envio a Cordoba?

* La amoladora tiene garantia?

* Venden a cualquier persona o solo a negocios?

* Necesito hablar con mi vendedor

In [23]:
#Prompt: Podrias crear un ciclo con las preguntas definidas en la celda anterior para evaluar el funcionamiento de los dos modelos?
preguntas_prueba = [
    "Tienen tornillos en stock?",
    "A que hora abren?",
    "Como hago para pagar una deuda?",
    "Me pueden mandar el pedido a otra dirección?",
    "Quien gano el partido de ayer?",
    "Que mecha uso para agujerear hormigon?",
    "Cuanto tarda el envio a Cordoba?",
    "La amoladora tiene garantia?",
    "Venden a cualquier persona o solo a negocios?",
    "Necesito hablar con mi vendedor",
]

for consulta in preguntas_prueba: #se prueba cada consulta en los dos chatbots
    print("=" * 80)
    print(f"Consulta: {consulta}\n")
    print("TF-IDF:")
    print(chatbot_tfidf(consulta), "\n")
    print("Embeddings:")
    print(chatbot_embeddings(consulta))

Consulta: Tienen tornillos en stock?

TF-IDF:
Pregunta más parecida: ¿Hay stock de tornillos y bulones? (similitud: 0.45)
Sí, la tornillería y bulonería en medidas estándar tiene stock permanente. Las medidas especiales se piden a fábrica. 

Embeddings:
Pregunta más parecida: ¿Hay stock de tornillos y bulones? (similitud: 0.75)
Sí, la tornillería y bulonería en medidas estándar tiene stock permanente. Las medidas especiales se piden a fábrica.
Consulta: A que hora abren?

TF-IDF:
Pregunta más parecida: ¿Cómo informo un pago que ya hice? (similitud: 0.38)
Enviá el comprobante a Cobranzas indicando tu razón social y qué facturas cancela ese pago. 

Embeddings:
Pregunta más parecida: ¿A quién le pago una factura vencida? (similitud: 0.64)
Al área de Cobranzas, que te indica los medios de pago disponibles: transferencia, cheque o cobrador.
Consulta: Como hago para pagar una deuda?

TF-IDF:
Pregunta más parecida: ¿Cómo hago para devolver un producto? (similitud: 0.37)
Las devoluciones se co

Ambos chatbots acertaron las consultas 1 y 10, que compartían palabras clave con preguntas del dataset. Además, TF-IDF acertó la 6 y la 7, y embeddings acertó la 9.

En los errores, TF-IDF se guió por palabras poco importantes ("que", "hago", "me", "el", "de") o por una palabra clave que aparecía en otra pregunta, como "amoladora" en la consulta 8. El chatbot con embeddings dio similitudes muy parecidas para todas las preguntas (entre 0.62 y 0.82), por lo que le cuesta diferenciar la respuesta correcta: su similitud más alta (0.82, en la consulta 6) fue justamente una respuesta incorrecta. Aun así, en las consultas 3 y 7 se acercó al tema correcto.

Ninguno de los dos puede reconocer cuando una pregunta está fuera de tema: siempre devuelven alguna respuesta, incluso para la consulta 5.

# Apartado 5:

## 5) Conclusiones

TF-IDF acertó 4 de 10 consultas y embeddings 3 de 10.

TF-IDF funciona bien cuando la consulta repite palabras clave del dataset (stock, mecha, vendedor), pero falla con sinónimos y palabras sin tilde, y se confunde por palabras de relleno ("que", "hago", "me"). Por ejemplo, confundió la garantía de la amoladora con la pregunta del disco para amoladora.

Embeddings entiende mejor algunas preguntas dichas de otra forma ("venden a cualquier persona" la relacionó con consumidor final), pero da similitudes muy parecidas para todo y a veces elige preguntas sin relación, como la garantía con la dirección de entrega.

Ninguno de los dos reconoce las preguntas fuera de tema: siempre responden algo.
